In [18]:
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV


In [2]:
df=pd.read_csv('cleaned_credit_risk_scoring.csv')
df

,Age,Income_Monthly,Loan_Amount_Requested,Credit_History_Length_Years,Existing_Loans_Count,Past_Defaults,Approved,Employment_Type_Freelancer,Employment_Type_Salaried,Employment_Type_Self-Employed
0,60,52841,1786857,13,3,0,0,False,True,False
1,50,85640,549137,6,2,0,0,False,False,False
2,36,180983,706884,16,4,0,1,False,False,False
3,64,153429,674712,11,3,1,1,False,False,True
4,29,178551,1900056,19,3,0,1,False,False,True
...,...,...,...,...,...,...,...,...,...,...
495,30,136101,1937106,7,2,1,0,False,False,False
496,45,176595,1769187,19,0,2,1,False,True,False
497,56,103861,696701,2,4,3,0,False,False,False
498,56,101740,836958,15,3,2,1,False,False,True


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   Age                            500 non-null    int64
 1   Income_Monthly                 500 non-null    int64
 2   Loan_Amount_Requested          500 non-null    int64
 3   Credit_History_Length_Years    500 non-null    int64
 4   Existing_Loans_Count           500 non-null    int64
 5   Past_Defaults                  500 non-null    int64
 6   Approved                       500 non-null    int64
 7   Employment_Type_Freelancer     500 non-null    bool 
 8   Employment_Type_Salaried       500 non-null    bool 
 9   Employment_Type_Self-Employed  500 non-null    bool 
dtypes: bool(3), int64(7)
memory usage: 28.9 KB


In [4]:
df1 = df[:350]
df2 = df[350:]
df1,df2

(     Age  Income_Monthly  Loan_Amount_Requested  Credit_History_Length_Years  \
 0     60           52841                1786857                           13   
 1     50           85640                 549137                            6   
 2     36          180983                 706884                           16   
 3     64          153429                 674712                           11   
 4     29          178551                1900056                           19   
 ..   ...             ...                    ...                          ...   
 345   60           88686                1174289                           14   
 346   50           82661                1841199                            3   
 347   63           59247                 538972                           10   
 348   47          162820                1027965                           18   
 349   56          120113                1173994                           14   
 
      Existing_Loans_Count

In [8]:
df1.shape, df2.shape

((350, 10), (150, 10))

In [9]:
df1.to_csv('train.csv', index=False)
df2.to_csv('test.csv', index=False)

In [26]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix,accuracy_score,roc_auc_score

In [ ]:

df1

,Age,Income_Monthly,Loan_Amount_Requested,Credit_History_Length_Years,Existing_Loans_Count,Past_Defaults,Approved,Employment_Type_Freelancer,Employment_Type_Salaried,Employment_Type_Self-Employed
0,60,52841,1786857,13,3,0,0,False,True,False
1,50,85640,549137,6,2,0,0,False,False,False
2,36,180983,706884,16,4,0,1,False,False,False
3,64,153429,674712,11,3,1,1,False,False,True
4,29,178551,1900056,19,3,0,1,False,False,True
...,...,...,...,...,...,...,...,...,...,...
345,60,88686,1174289,14,1,2,1,False,True,False
346,50,82661,1841199,3,3,1,0,True,False,False
347,63,59247,538972,10,4,2,0,False,False,True
348,47,162820,1027965,18,3,2,1,False,False,True


In [52]:

X = df1.drop(columns=['Approved','Loan_Amount_Requested','Age'],axis=1)
Y = df1['Approved']
X.shape,Y.shape

((350, 7), (350,))

In [53]:
sd = StandardScaler()
X_scaled = sd.fit_transform(X)

In [54]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, Y, test_size=0.2, random_state=42
)

In [55]:
param_grid = {
    "n_estimators": [50,100,150, 200],
    "max_depth": [2,3,4,5,6,7],
    "min_samples_split": [1,2,3,4, 5],
    "min_samples_leaf": [1, 2,3]
}

In [56]:

rf = RandomForestClassifier(random_state=42)
grid = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,                 # 5-fold cross-validation
    scoring="roc_auc",    # better than accuracy for loans
    n_jobs=-1,            # use all CPU cores
    verbose=2
)

grid.fit(X_train, y_train)

Fitting 5 folds for each of 360 candidates, totalling 1800 fits


C:\Users\kk673\AppData\Roaming\Python\Python312\site-packages\sklearn\model_selection\_validation.py:540: FitFailedWarning: 
360 fits failed out of a total of 1800.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
360 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\kk673\AppData\Roaming\Python\Python312\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\kk673\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py", line 1466, in wrapper
    estimator._validate_params()
  File "C:\Users\kk673\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py", line 666, in _validate_params
    val

GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [2, 3, 4, 5, 6, 7],
                         'min_samples_leaf': [1, 2, 3],
                         'min_samples_split': [1, 2, 3, 4, 5],
                         'n_estimators': [50, 100, 150, 200]},
             scoring='roc_auc', verbose=2)

In [57]:
best_model = grid.best_estimator_

print("Best Parameters:", grid.best_params_)

Best Parameters: {'max_depth': 3, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}


In [58]:
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred))

Accuracy: 0.7857142857142857
ROC-AUC: 0.9013157894736842
              precision    recall  f1-score   support

           0       0.79      0.72      0.75        32
           1       0.78      0.84      0.81        38

    accuracy                           0.79        70
   macro avg       0.79      0.78      0.78        70
weighted avg       0.79      0.79      0.78        70



In [59]:
confusion_matrix(y_test, y_pred)

array([[23,  9],
       [ 6, 32]], dtype=int64)

In [61]:
x2 = df2.drop(columns=['Approved','Loan_Amount_Requested','Age'],axis=1)
y2 = df2['Approved']

In [64]:
x2_scaled = sd.transform(x2)
y2_pred = best_model.predict(x2_scaled)
print("Test Set Accuracy:", accuracy_score(y2, y2_pred))
print("Test Set ROC-AUC:", roc_auc_score(y2, best_model.predict_proba(x2_scaled)[:, 1]))
print(classification_report(y2, y2_pred))


Test Set Accuracy: 0.88
Test Set ROC-AUC: 0.9635642135642135
              precision    recall  f1-score   support

           0       0.92      0.86      0.89        84
           1       0.83      0.91      0.87        66

    accuracy                           0.88       150
   macro avg       0.88      0.88      0.88       150
weighted avg       0.88      0.88      0.88       150



In [65]:
import joblib

# Save trained model
joblib.dump(best_model, "loan_model.pkl")
joblib.dump(sd, "scaler.pkl")

['scaler.pkl']